# 🎯 Module 13: GANs — Generative Adversarial Networks

> **Stage:** Deep Learning  
> **Level:** Intermediate → Job-Ready  
> **Module:** Generative Adversarial Networks (GANs)  
> **Framework:** PyTorch  
> **Date:** 13 June 2026  
> **Previous:** 12 — Autoencoders  
> **Next:** 14 — Computer Vision with Deep Learning

---

Autoencoders **reconstruct** data. GANs **generate** brand-new data by pitting two neural networks against each other in a game.

In this notebook we'll cover:

- What GANs are and why they matter
- **Generator** and **Discriminator**
- **Adversarial training** and the minimax objective
- Building a **fully-connected GAN** for MNIST
- **DCGAN** — convolutional GANs
- **Mode collapse** and training instability
- **WGAN** — why it exists
- GANs vs Autoencoders

## 📚 Learning Objectives

By the end of this notebook you should be able to:

- Explain what a GAN is and why it's *adversarial*
- Describe the roles of **Generator** and **Discriminator**
- Write the minimax objective from memory
- Explain the **non-saturating generator loss**
- Build and train a simple GAN in PyTorch
- Explain `detach()` in the discriminator step
- Build a **DCGAN** for image generation
- Diagnose **mode collapse**
- Explain **WGAN** at a high level
- Compare GANs with autoencoders

---
## ⚙️ 0. Setup & Imports

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.utils import make_grid

import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else
                      'mps'  if torch.backends.mps.is_available() else 'cpu')
print(f'Using device: {device}')
print(f'PyTorch version: {torch.__version__}')

---
## 🧠 1. What Are GANs?

**Generative Adversarial Networks (GANs)** are deep learning models used to **generate new data** that resembles real training data.

### Applications

- Generate realistic images
- Create synthetic faces
- Image-to-image translation (pix2pix, CycleGAN)
- Data augmentation
- Generate artwork / textures

### Two networks competing

| Network | Job |
| --- | --- |
| **Generator (G)** | Creates fake samples |
| **Discriminator (D)** | Determines real vs fake |

### The counterfeiter & detective analogy

```text
Random Noise
     ↓
 Generator
     ↓
 Fake Image ──────┐
                  ↓
             Discriminator
                  ↑
 Real Image ──────┘
                  ↓
            Real / Fake
```

The Generator improves by **trying to fool** the Discriminator.
The Discriminator improves by **catching** the Generator.
They push each other to become better.

---
## 🎨 2. The Generator

The Generator receives **random noise**:

$$z \sim P(z)$$

and produces a synthetic sample:

$$G(z)$$

### Example

```text
Random vector
[0.2, -1.1, 0.7, ...]
       ↓
   Generator
       ↓
   Fake Image
```

### Goal

> Make generated samples look realistic enough that the Discriminator classifies them as **real**.

---
## 🕵️ 3. The Discriminator

The Discriminator receives either:

- A **real** training sample
- A **generated** (fake) sample

and predicts whether it's real:

$$D(x) \rightarrow \text{probability(real)}$$

### Example

```text
Real image → Discriminator → 0.94
Fake image → Discriminator → 0.08
```

During training, the Discriminator learns to distinguish real from fake, while the Generator learns to produce better fakes.

---
## ⚔️ 4. Adversarial Training

GAN training is a **two-player game**:

$$\min_G \max_D V(D, G)$$

### The original objective

$$\min_G \max_D \; \mathbb{E}_{x \sim p_{data}}[\log D(x)] + \mathbb{E}_{z \sim p_z}[\log(1 - D(G(z)))]$$

Don't worry about memorizing it yet — focus on the **competition**:

```text
Generator:
  "Make fake samples realistic."

Discriminator:
  "Detect fake samples."

Generator improves
      ↓
Discriminator improves
      ↓
Generator improves
      ↓
   ...
```

Ideally, the Generator eventually produces samples that are difficult to distinguish from real data.

### The non-saturating generator loss

In practice, the Generator is trained with a **non-saturating loss** which has better gradients:

$$L_G = -\mathbb{E}_z[\log D(G(z))]$$

---
## 📉 5. GAN Loss Intuition

### For the Discriminator (binary cross-entropy)

```text
Real sample: D(x) = 0.9     target = 1
Fake sample: D(G(z)) = 0.1  target = 0
```

The Discriminator wants to push `D(real) → 1` and `D(fake) → 0`.

### For the Generator

The Generator wants the Discriminator to output `1` on fake samples.

$$L_G = -\mathbb{E}_z[\log D(G(z))]$$

Minimizing this loss is equivalent to **maximizing** the Discriminator's predicted probability that generated samples are real.

---
## 🐍 6. Building a Simple GAN in PyTorch

For MNIST, each image is `28 × 28 = 784` pixels.

- The Generator takes a **noise vector** and outputs 784 numbers.
- The Discriminator takes 784 numbers and outputs a single probability.

Note the `tanh` activation on the Generator output — it maps values to `[-1, 1]`, which is the range MNIST will be normalized to.

In [ ]:
class Generator(nn.Module):
    def __init__(self, noise_dim=100):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(noise_dim, 256), nn.ReLU(),
            nn.Linear(256, 512),       nn.ReLU(),
            nn.Linear(512, 784),       nn.Tanh(),   # output in [-1, 1]
        )

    def forward(self, z):
        return self.model(z)


class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(784, 512), nn.LeakyReLU(0.2),
            nn.Linear(512, 256), nn.LeakyReLU(0.2),
            nn.Linear(256, 1),   nn.Sigmoid(),   # probability of being real
        )

    def forward(self, x):
        return self.model(x)


# Shape check
G = Generator(noise_dim=100)
D = Discriminator()
z = torch.randn(4, 100)
print('Noise:', z.shape)
print('Fake :', G(z).shape)
print('D(fake):', D(G(z)).shape)
print('Params G:', sum(p.numel() for p in G.parameters()))
print('Params D:', sum(p.numel() for p in D.parameters()))

---
## 📦 7. Loading MNIST

We normalize to `[-1, 1]` so it matches the Generator's `tanh` output.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),          # → [-1, 1]
    transforms.Lambda(lambda x: x.view(-1)),       # flatten → 784
])

train_ds = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
loader   = DataLoader(train_ds, batch_size=128, shuffle=True, drop_last=True, num_workers=2)

print('Train:', len(train_ds))
print('Sample shape:', train_ds[0][0].shape)
print('Sample range:', train_ds[0][0].min().item(), '→', train_ds[0][0].max().item())

---
## 🔁 8. GAN Training Loop

Training alternates between the two networks:

### Step 1 — Train the Discriminator

```text
Real images → D → target 1
Fake images → D → target 0
Update only D.
```

### Step 2 — Train the Generator

```text
Noise → G → fake image → D
                         ↓
                    should be 1
Update only G.
```

⚠️ **Critical:** during Step 1, we call `G(z).detach()` so the fake images are treated as **constants** — the Discriminator's backward pass won't touch the Generator.

In [ ]:
def train_gan(G, D, loader, opt_G, opt_D, criterion, noise_dim, epochs=15, device='cpu'):
    G.to(device); D.to(device)
    history = {'loss_D': [], 'loss_G': []}
    sample_z = torch.randn(64, noise_dim, device=device)   # fixed noise for tracking progress

    for epoch in range(1, epochs + 1):
        epoch_D, epoch_G, n = 0.0, 0.0, 0

        for real, _ in loader:
            real = real.to(device)
            B = real.size(0)

            # ---------------- Train Discriminator ----------------
            D.zero_grad()

            # Real
            real_pred = D(real)
            loss_real = criterion(real_pred, torch.ones_like(real_pred))

            # Fake (detached!)
            z = torch.randn(B, noise_dim, device=device)
            fake = G(z).detach()
            fake_pred = D(fake)
            loss_fake = criterion(fake_pred, torch.zeros_like(fake_pred))

            loss_D = loss_real + loss_fake
            loss_D.backward()
            opt_D.step()

            # ---------------- Train Generator ----------------
            G.zero_grad()
            z = torch.randn(B, noise_dim, device=device)
            fake = G(z)
            pred = D(fake)

            # Non-saturating loss: pretend fakes are real (target = 1)
            loss_G = criterion(pred, torch.ones_like(pred))
            loss_G.backward()
            opt_G.step()

            epoch_D += loss_D.item() * B
            epoch_G += loss_G.item() * B
            n += B

        history['loss_D'].append(epoch_D / n)
        history['loss_G'].append(epoch_G / n)
        print(f'Epoch {epoch:2d}/{epochs} | loss_D={history["loss_D"][-1]:.4f} | loss_G={history["loss_G"][-1]:.4f}')

    return history, sample_z

print('Training function ready.')

In [ ]:
# Train the GAN
torch.manual_seed(42)
noise_dim = 100
G = Generator(noise_dim=noise_dim)
D = Discriminator()

criterion = nn.BCELoss()
opt_G = optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))
opt_D = optim.Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.999))

history, fixed_z = train_gan(
    G, D, loader, opt_G, opt_D, criterion,
    noise_dim=noise_dim, epochs=15, device=device,
)

In [ ]:
# Plot generator & discriminator losses
plt.figure(figsize=(9, 4))
plt.plot(history['loss_D'], label='Discriminator')
plt.plot(history['loss_G'], label='Generator')
plt.title('GAN losses over training')
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# Visualize generated samples
def show_samples(G, z, title='Generated samples', n=64):
    G.eval()
    with torch.no_grad():
        fake = G(z).cpu().view(-1, 1, 28, 28)
    grid = make_grid(fake, nrow=8, normalize=True)
    plt.figure(figsize=(6, 6))
    plt.imshow(grid.permute(1, 2, 0).numpy(), cmap='gray')
    plt.title(title)
    plt.axis('off')
    plt.tight_layout(); plt.show()

show_samples(G, fixed_z, title='After 15 epochs')

In [ ]:
# Show progress: fresh samples every few steps
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for i, ax in enumerate(axes):
    z = torch.randn(16, noise_dim, device=device)
    with torch.no_grad():
        fake = G(z).cpu().view(-1, 1, 28, 28)
    grid = make_grid(fake, nrow=4, normalize=True)
    ax.imshow(grid.permute(1, 2, 0).numpy(), cmap='gray')
    ax.set_title(f'batch {i+1}')
    ax.axis('off')
plt.suptitle('Fresh samples from the trained Generator')
plt.tight_layout(); plt.show()

---
## 🖼️ 9. DCGAN — Deep Convolutional GAN

The fully-connected GAN works but produces blurry samples.
**DCGAN** replaces dense layers with **convolutions**, which are far better suited to images.

### Generator (DCGAN)

```text
Noise
  ↓
Transpose Convolutions
  ↓
Upsampling
  ↓
Generated Image
```

### Discriminator (DCGAN)

```text
Image
  ↓
Convolution Layers
  ↓
Feature Extraction
  ↓
Real / Fake
```

### DCGAN architectural guidelines

- Replace pooling with **strided convolutions** (D) / **fractional-strided convolutions** (G)
- Use **BatchNorm** in both networks (except G output & D input)
- Remove fully-connected hidden layers
- Use **ReLU** in G (except output → **tanh**)
- Use **LeakyReLU** in D

In [ ]:
class DCGenerator(nn.Module):
    def __init__(self, noise_dim=100, feat=64):
        super().__init__()
        self.net = nn.Sequential(
            # z: [B, noise, 1, 1] → [B, feat*4, 7, 7]
            nn.ConvTranspose2d(noise_dim, feat * 4, 7, 1, 0, bias=False),
            nn.BatchNorm2d(feat * 4),
            nn.ReLU(True),
            # → [B, feat*2, 14, 14]
            nn.ConvTranspose2d(feat * 4, feat * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(feat * 2),
            nn.ReLU(True),
            # → [B, 1, 28, 28]
            nn.ConvTranspose2d(feat * 2, 1, 4, 2, 1, bias=False),
            nn.Tanh(),
        )

    def forward(self, z):
        return self.net(z.view(z.size(0), -1, 1, 1))


class DCDiscriminator(nn.Module):
    def __init__(self, feat=64):
        super().__init__()
        self.net = nn.Sequential(
            # [B, 1, 28, 28] → [B, feat, 14, 14]
            nn.Conv2d(1, feat, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            # → [B, feat*2, 7, 7]
            nn.Conv2d(feat, feat * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(feat * 2),
            nn.LeakyReLU(0.2, inplace=True),
            # → [B, 1, 1, 1] → scalar
            nn.Conv2d(feat * 2, 1, 7, 1, 0, bias=False),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.net(x).view(-1, 1)


# Shape check
G_dc = DCGenerator(noise_dim=100)
D_dc = DCDiscriminator()
z = torch.randn(4, 100)
x_fake = G_dc(z)
print('Noise       :', z.shape)
print('Fake image  :', x_fake.shape)
print('D(fake)     :', D_dc(x_fake).shape)
print('G params    :', sum(p.numel() for p in G_dc.parameters()))
print('D params    :', sum(p.numel() for p in D_dc.parameters()))

In [ ]:
# Load MNIST in image form for the DCGAN
dc_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])
dc_train = datasets.MNIST(root='./data', train=True, download=True, transform=dc_transform)
dc_loader = DataLoader(dc_train, batch_size=128, shuffle=True, drop_last=True, num_workers=2)

def train_dcgan(G, D, loader, opt_G, opt_D, criterion, noise_dim, epochs=10, device='cpu'):
    G.to(device); D.to(device)
    history = {'loss_D': [], 'loss_G': []}
    fixed_z = torch.randn(64, noise_dim, device=device)

    for epoch in range(1, epochs + 1):
        eD, eG, n = 0.0, 0.0, 0
        for real, _ in loader:
            real = real.to(device)
            B = real.size(0)

            # --- Discriminator ---
            D.zero_grad()
            real_pred = D(real)
            loss_real = criterion(real_pred, torch.ones_like(real_pred))

            z = torch.randn(B, noise_dim, device=device)
            fake = G(z).detach()
            fake_pred = D(fake)
            loss_fake = criterion(fake_pred, torch.zeros_like(fake_pred))

            loss_D = loss_real + loss_fake
            loss_D.backward()
            opt_D.step()

            # --- Generator ---
            G.zero_grad()
            z = torch.randn(B, noise_dim, device=device)
            pred = D(G(z))
            loss_G = criterion(pred, torch.ones_like(pred))
            loss_G.backward()
            opt_G.step()

            eD += loss_D.item() * B
            eG += loss_G.item() * B
            n += B

        history['loss_D'].append(eD / n)
        history['loss_G'].append(eG / n)
        print(f'Epoch {epoch:2d}/{epochs} | loss_D={history["loss_D"][-1]:.4f} | loss_G={history["loss_G"][-1]:.4f}')

    return history, fixed_z

torch.manual_seed(42)
G_dc = DCGenerator(noise_dim=100)
D_dc = DCDiscriminator()
criterion = nn.BCELoss()
opt_G = optim.Adam(G_dc.parameters(), lr=2e-4, betas=(0.5, 0.999))
opt_D = optim.Adam(D_dc.parameters(), lr=2e-4, betas=(0.5, 0.999))

hist_dc, fixed_z_dc = train_dcgan(G_dc, D_dc, dc_loader, opt_G, opt_D, criterion,
                                  noise_dim=100, epochs=10, device=device)

In [ ]:
# Visualize DCGAN samples
G_dc.eval()
with torch.no_grad():
    fake = G_dc(fixed_z_dc).cpu()
grid = make_grid(fake, nrow=8, normalize=True)
plt.figure(figsize=(6, 6))
plt.imshow(grid.permute(1, 2, 0).numpy(), cmap='gray')
plt.title('DCGAN samples after 10 epochs')
plt.axis('off')
plt.tight_layout(); plt.show()

---
## 💥 10. Major GAN Problems

### Mode Collapse

The Generator produces **very similar samples repeatedly**:

```text
Expected:
A B C D E F

Generated:
A A A A A A
```

The Generator finds one type of output that successfully fools the Discriminator and gets stuck there.

### Training Instability

GANs are notoriously difficult to train because **both networks change simultaneously**:

- Discriminator becomes too strong → Generator gets no useful gradient
- Discriminator becomes too weak → Generator has no meaningful challenge
- Loss curves oscillate
- Poor-quality samples emerge
- Mode collapse

### ⚠️ Key insight

> **GAN loss curves alone do NOT tell you whether generation quality is improving.**
>
> You must also **look at the samples**.

In [ ]:
# Quick demo: monitor sample diversity across epochs
# (A truly collapsed GAN would show nearly identical samples)
G_dc.eval()
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for ax in axes:
    z = torch.randn(8, 100, device=device)
    with torch.no_grad():
        samples = G_dc(z).cpu()
    grid = make_grid(samples, nrow=8, normalize=True)
    ax.imshow(grid.permute(1, 2, 0).numpy(), cmap='gray')
    ax.axis('off')
plt.suptitle('Sample diversity check — 5 random batches')
plt.tight_layout(); plt.show()

---
## 🌊 11. WGAN — Wasserstein GAN

**WGAN** changes the training objective to provide **more useful gradients** and improve **training stability**.

### High-level idea

```text
Standard GAN
→ Classification-style real/fake objective
→ JS divergence (saturates when distributions don't overlap)

WGAN
→ Measures a smoother distance between distributions (Wasserstein)
→ Gradients remain meaningful even when D is very strong
```

### Key changes

| | Standard GAN | WGAN |
| --- | --- | --- |
| D output | Sigmoid probability | **Real number (critic)** |
| Loss | BCE | Wasserstein distance |
| Constraint | None | Weight clipping or gradient penalty |
| Stability | Often unstable | Usually more stable |

### Why it matters

You should understand **why WGAN exists** before diving into its math: the original GAN objective gives **vanishing gradients** when the Discriminator becomes too good, and WGAN fixes this.

---
## 🆚 12. GANs vs Autoencoders

| Feature | Autoencoder | GAN |
| --- | --- | --- |
| Main goal | Reconstruction | **Generation** |
| Architecture | Encoder + Decoder | Generator + Discriminator |
| Latent representation | Explicit | **Noise input** |
| Training | Reconstruction loss | **Adversarial loss** |
| Typical output | Reconstructed data | **New synthetic data** |
| Training difficulty | Usually easier | **Usually harder** |

### When to use which

- **Autoencoder** → compression, denoising, anomaly detection
- **GAN** → realistic generation, image translation, data augmentation

---
## 🚨 13. Common GAN Mistakes

### ❌ 1. Updating both networks in one backward pass
They must be trained in **separate steps** with separate optimizers.

### ❌ 2. Forgetting `detach()` for fake samples in D training
Without it, the Generator gets gradients from the Discriminator step.

### ❌ 3. Using incorrect real/fake labels
Real → 1, Fake → 0 (for D); Fake → 1 (for G's non-saturating loss).

### ❌ 4. Applying `softmax` to the Discriminator output
The Discriminator outputs **one** probability — use **sigmoid**, not softmax.

### ❌ 5. Ignoring input normalization
Match the Generator output range (`tanh` → `[-1, 1]`).

### ❌ 6. Judging GAN quality only from loss values
**Always look at the generated samples.**

### ❌ 7. Making the Discriminator overwhelmingly stronger
If D is too good, G gets no useful gradient. Balance the two.

### ❌ 8. Expecting stable training immediately
GANs are notoriously sensitive to hyperparameters — Adam with `betas=(0.5, 0.999)` and `lr=2e-4` is a good default.

---
## 🏋️ 14. Hands-On Exercises

### Exercise 1 — Noise Dimension Sweep
Try `noise_dim ∈ {10, 50, 100, 256}`. How does generation quality change?

### Exercise 2 — Learning Rate Tuning
Try `lr ∈ {1e-4, 2e-4, 5e-4}`. Which gives the most stable training?

### Exercise 3 — Deeper Generator
Add another hidden layer. Does sample quality improve or does training become unstable?

### Exercise 4 — Label Smoothing
Instead of `1.0` for real samples, use `0.9`. Does training stabilize?

### Exercise 5 — Track Generated Images Over Epochs
Save samples every epoch and build a GIF or grid to visualize progress.

### Exercise 6 — CIFAR-10 DCGAN
Adapt the DCGAN to 3-channel 32×32 images. What changes are needed?

### Exercise 7 — Mode Collapse Detection
Compute pairwise distances between generated samples. Very low average distance is a signal of mode collapse.

---
## 🎤 15. Interview Questions

1. **What is a GAN?**  
   A pair of networks — Generator and Discriminator — trained in an adversarial game to generate realistic data.

2. **What are Generator and Discriminator?**  
   G creates fake samples from noise; D classifies samples as real or fake.

3. **Why are GANs called adversarial?**  
   Because the two networks have opposing objectives.

4. **What does the Generator optimize?**  
   It minimizes the probability that D correctly identifies its outputs as fake (non-saturating loss).

5. **What does the Discriminator optimize?**  
   It maximizes the probability of correctly classifying real vs fake.

6. **What is mode collapse?**  
   When G produces very similar outputs regardless of the input noise.

7. **Why are GANs difficult to train?**  
   The two networks change simultaneously — the game is inherently unstable.

8. **What is DCGAN?**  
   A convolutional GAN with specific architectural guidelines for stable image generation.

9. **Why is `detach()` used while training D?**  
   To prevent the Generator from being updated during D's backward pass.

10. **What problem does WGAN try to address?**  
    Training instability and vanishing gradients in the original GAN objective.

---
## ✅ 16. Knowledge Check

- [ ] Explain what a GAN is
- [ ] Describe the Generator and Discriminator
- [ ] Explain why GANs are adversarial
- [ ] Write the minimax objective
- [ ] Explain the non-saturating generator loss
- [ ] Build and train a fully-connected GAN
- [ ] Use `detach()` correctly
- [ ] Build a DCGAN
- [ ] Diagnose mode collapse
- [ ] Explain WGAN at a high level
- [ ] Compare GANs and autoencoders
- [ ] Inspect generated samples, not just losses

---
## 🏁 17. Key Takeaways

### GAN structure

```text
GAN
│
├── Generator
│     z → Fake sample
│
└── Discriminator
      x → Real/Fake probability
```

### The training game

$$\min_G \max_D \; \mathbb{E}_{x} [\log D(x)] + \mathbb{E}_z [\log(1 - D(G(z)))]$$

### Practical generator loss

$$L_G = -\mathbb{E}_z [\log D(G(z))]$$

### The GAN family

```text
GAN
 ├── DCGAN       (convolutional)
 ├── WGAN        (stable objective)
 ├── WGAN-GP     (gradient penalty)
 ├── Pix2Pix     (image translation)
 ├── CycleGAN    (unpaired translation)
 ├── StyleGAN    (high-fidelity faces)
 └── ...
```

### 🧭 Golden Rule

> **GAN loss curves alone do not tell you whether training is going well. Always inspect generated samples visually — and expect instability.**

---

### 🔗 What's Next?

**Module 14 — Computer Vision with Deep Learning**

GANs generate images. The next module focuses on the **full computer-vision pipeline** — from **classification** and **detection** to **segmentation**, with metrics, transfer learning, and deployment considerations.